# P2 · N1 — Census and Prompt Payloads

**Paper 2 — MARQ-Bench: Machine-Authored Data Quality**

This notebook turns the N0 splits into the actual text that will be sent to the
models.

1. Reload the corpora and restore the N0 authoring/evaluation splits
2. Check A3 documentation coverage
3. Profile the **authoring split only** to build the census
4. Assemble the A2–A5 prompt payloads for every corpus
5. Run leakage checks and eyeball the output

**Prerequisite:** N0 must have completed. This notebook reads its checkpoints.

**Resumability:** every step is checkpointed. Re-run top to bottom after a
disconnect. To rebuild a step, delete its file from `checkpoints/`.

**Runtime:** ~5–10 minutes, mostly the Excel and Parquet loads. No GPU. No API keys.

## 1 · Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Setup

Defines everything the rest of the notebook uses. Safe to re-run at any point —
that is the fix if you hit a `NameError` after a runtime restart.

In [ ]:
import sys, json, datetime
from pathlib import Path

ROOT        = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES     = ROOT / 'Notebooks'
CHECKPOINTS = ROOT / 'checkpoints'
ARTIFACTS   = ROOT / 'artifacts'

assert ROOT.exists(),    f'project folder not found: {ROOT}'
assert MODULES.exists(), f'modules folder not found: {MODULES}'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

import llmauth_census as C
import llmauth_prompts as P
import llmauth_docs as D
import llmauth_checkpoint as CK

ckpt = CK.Checkpoint(CHECKPOINTS)

PATTERNS = {
    'bank_marketing':   ['bankfull', 'bank-full', 'bank_full'],
    'diabetes_130us':   ['diabetic_data', 'diabetes'],
    'online_retail_ii': ['online_retail', 'online retail', 'retail'],
    'nyc_tlc_yellow':   ['yellow_tripdata'],
}
CONDITIONS = ('A2', 'A3', 'A4', 'A5')

def discover_data():
    files = [p for p in ROOT.rglob('*')
             if p.is_file() and p.suffix.lower() in ('.csv', '.parquet', '.xlsx')]
    found = {}
    for corpus, frags in PATTERNS.items():
        hits = [p for p in files if any(f in p.name.lower() for f in frags)]
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
        if hits:
            found[corpus] = hits[0]
    return found

print('census', C.CENSUS_VERSION, '| prompts', P.PROMPT_VERSION,
      '| docs', D.DOCS_VERSION, '| checkpoint', CK.CHECKPOINT_VERSION)

census 1.0.0 | prompts 1.0.0 | docs 1.0.0 | checkpoint 1.0.0


## 3 · Confirm N0 finished

If any split is missing, go back and run N0 before continuing.

In [ ]:
required = [f'split_{c}' for c in PATTERNS] + ['n0_provenance']
missing = [n for n in required if not ckpt.exists(n, 'json')]

for n in required:
    print(('OK  ' if ckpt.exists(n, 'json') else '!!  ') + n + '.json')

assert not missing, f'N0 has not completed: missing {missing}'
print('\nN0 checkpoints present.')

OK  split_bank_marketing.json
OK  split_diabetes_130us.json
OK  split_online_retail_ii.json
OK  split_nyc_tlc_yellow.json
OK  n0_provenance.json

N0 checkpoints present.


## 4 · Reload corpora and restore the splits

`load_corpus` re-reads each file with its registered configuration, so the
sentinels and the full Retail workbook come back intact. The split indices come
from N0, so the partition is identical — no re-randomisation.

In [ ]:
DATA_PATHS = discover_data()
corpora, authoring, evaluation = {}, {}, {}

for corpus, path in DATA_PATHS.items():
    df, prov = C.load_corpus(corpus, path)
    corpora[corpus] = df
    split = ckpt.step(f'split_{corpus}', 'json',
                      lambda: (_ for _ in ()).throw(RuntimeError('run N0 first')))[0]
    authoring[corpus]  = df.loc[split['authoring_index']]
    evaluation[corpus] = df.loc[split['evaluation_index']]
    print(f'{corpus:<18} total {len(df):>9,}  authoring {len(authoring[corpus]):>8,}  '
          f'evaluation {len(evaluation[corpus]):>8,}')

# the split must still be disjoint after reload
for c in corpora:
    assert not (set(authoring[c].index) & set(evaluation[c].index)), f'{c}: splits overlap'
print('\nOK  splits restored and disjoint.')

  [cached] split_bank_marketing.json  (written 2026-08-08T03:39:10+00:00)
bank_marketing     total    45,211  authoring    9,042  evaluation   36,169
  [cached] split_diabetes_130us.json  (written 2026-08-08T03:39:11+00:00)
diabetes_130us     total   101,766  authoring   20,353  evaluation   81,413
  [cached] split_online_retail_ii.json  (written 2026-08-08T14:34:08+00:00)
online_retail_ii   total 1,067,371  authoring  213,474  evaluation  853,897
  [cached] split_nyc_tlc_yellow.json  (written 2026-08-08T03:39:14+00:00)
nyc_tlc_yellow     total 4,090,836  authoring  200,000  evaluation  800,000

OK  splits restored and disjoint.


## 5 · Documentation coverage for condition A3

A3 supplies the author with published field documentation. The condition is
only fair if the documentation covers the schema — an undocumented column
quietly makes A3 equivalent to A2 for that column.

In [ ]:
cov = {}
for corpus, df in corpora.items():
    r = D.coverage_report(corpus, list(df.columns))
    cov[corpus] = r
    flag = 'OK ' if r['coverage'] == 1.0 else '!! '
    print(f"{flag}{corpus:<18} {r['n_documented']:>2}/{r['n_columns']:<3} "
          f"= {r['coverage']:.0%}")
    if r['undocumented']:
        print(f"     undocumented: {r['undocumented']}")
    if r['orphaned_docs']:
        print(f"     docs for absent columns: {r['orphaned_docs']}")

_, _ = ckpt.step('a3_doc_coverage', 'json', lambda: cov,
                 code_version=D.DOCS_VERSION)

OK bank_marketing     17/17  = 100%
OK diabetes_130us     50/50  = 100%
OK online_retail_ii    8/8   = 100%
OK nyc_tlc_yellow     20/20  = 100%
  [cached] a3_doc_coverage.json  (written 2026-08-08T15:17:55+00:00)


## 6 · Build the census

Profiles the **authoring split only**. Rules are authored from this; every
metric is later computed on the disjoint evaluation split.

> **No cardinality gate.** Value-frequency lists are built for every column,
> numeric ones included. `pdays` has hundreds of distinct values, so a
> conventional profiler would emit no level list for it and the fact that most
> values are exactly `-1` would never reach the prompt. That omission would
> hollow out condition A4 and bias the study against its own hypothesis.

In [ ]:
facts = {}
for corpus in corpora:
    def build(corpus=corpus):
        f = C.profile_authoring_split(
            authoring[corpus], corpus,
            docs=D.get_docs(corpus),
            table_doc=D.get_table_doc(corpus),
        )
        return json.loads(json.dumps({
            'corpus_id': f.corpus_id, 'row_count': f.row_count,
            'table_doc': f.table_doc,
            'columns': [{'name': c.name, 'dtype': c.dtype, 'doc': c.doc,
                         'null_rate': c.null_rate,
                         'distinct_count': c.distinct_count,
                         'min_value': c.min_value, 'max_value': c.max_value,
                         'quantiles': c.quantiles,
                         'top_levels': [[v, n] for v, n in c.top_levels]}
                        for c in f.columns],
        }, default=str))

    payload, _ = ckpt.step(f'census_{corpus}', 'json', build,
                           code_version=C.CENSUS_VERSION)
    facts[corpus] = P.CorpusFacts(
        corpus_id=payload['corpus_id'], row_count=payload['row_count'],
        table_doc=payload['table_doc'],
        columns=[P.ColumnFacts(
            name=c['name'], dtype=c['dtype'], doc=c['doc'],
            null_rate=c['null_rate'], distinct_count=c['distinct_count'],
            min_value=c['min_value'], max_value=c['max_value'],
            quantiles=c['quantiles'],
            top_levels=[(v, n) for v, n in c['top_levels']],
        ) for c in payload['columns']],
    )

print()
for corpus, f in facts.items():
    no_levels = [c.name for c in f.columns if not c.top_levels]
    print(f'{corpus:<18} {len(f.columns):>2} columns, {f.row_count:>8,} authoring rows'
          + (f'   !! no level list: {no_levels}' if no_levels else ''))

  [cached] census_bank_marketing.json  (written 2026-08-08T15:17:55+00:00)
  [cached] census_diabetes_130us.json  (written 2026-08-08T15:17:58+00:00)
  [cached] census_online_retail_ii.json  (written 2026-08-08T15:18:01+00:00)
  [cached] census_nyc_tlc_yellow.json  (written 2026-08-08T15:18:02+00:00)

bank_marketing     17 columns,    9,042 authoring rows
diabetes_130us     50 columns,   20,353 authoring rows
online_retail_ii    8 columns,  213,474 authoring rows
nyc_tlc_yellow     20 columns,  200,000 authoring rows


## 7 · Sanity-check the census

Confirms the sentinel frequencies actually made it into the profile. These are
the statistics condition A4 exists to supply.

In [ ]:
def modal(corpus, column):
    col = {c.name: c for c in facts[corpus].columns}[column]
    if not col.top_levels:
        return None, None
    v, n = col.top_levels[0]
    return v, n / facts[corpus].row_count

for corpus, column in [('bank_marketing', 'pdays'),
                       ('bank_marketing', 'poutcome'),
                       ('diabetes_130us', 'max_glu_serum'),
                       ('diabetes_130us', 'weight'),
                       ('nyc_tlc_yellow', 'RatecodeID'),
                       ('nyc_tlc_yellow', 'payment_type')]:
    if corpus not in facts:
        continue
    v, rate = modal(corpus, column)
    print(f'{corpus:<18} {column:<16} modal={v!r:<12} {rate:.2%}' if rate is not None
          else f'{corpus:<18} {column:<16} !! no level list')

bank_marketing     pdays            modal=-1           82.36%
bank_marketing     poutcome         modal='unknown'    82.36%
diabetes_130us     max_glu_serum    modal='None'       94.71%
diabetes_130us     weight           modal='?'          96.91%
nyc_tlc_yellow     RatecodeID       modal=1.0          69.47%
nyc_tlc_yellow     payment_type     modal=1            66.74%


## 8 · Build the A2–A5 prompt payloads

The information ladder:

| | Payload |
|---|---|
| **A2** | schema only — column names and types |
| **A3** | schema + published documentation |
| **A4** | schema + census |
| **A5** | schema + census + 20 sampled rows |

`build_prompt` runs a leakage guard over the census text: it may report *that* a
value occurs and *how often*, never what it means. Published documentation in A3
is deliberately exempt — the whole point of A3 is to test whether documentation
is sufficient.

In [ ]:
SAMPLE_SEED = 20260807

def rows_for(corpus, n=200):
    return authoring[corpus].head(n).to_dict('records')

prompts = {}
for corpus in facts:
    for cond in CONDITIONS:
        b = P.build_prompt(cond, facts[corpus],
                           rows_for(corpus) if cond == 'A5' else None,
                           sample_seed=SAMPLE_SEED)
        prompts[(corpus, cond)] = b

def bundle_record(b):
    return {**b.provenance(), 'system': b.system, 'user': b.user,
            'user_chars': len(b.user), 'est_tokens': len(b.user) // 4}

snapshot, _ = ckpt.step(
    'prompt_payloads', 'json',
    lambda: {f'{c}|{k}': bundle_record(b) for (c, k), b in prompts.items()},
    code_version=P.PROMPT_VERSION)

print(f'{"corpus":<18} {"A2":>8} {"A3":>8} {"A4":>8} {"A5":>8}   (approx tokens)')
print('-' * 60)
for corpus in facts:
    row = [f'{len(prompts[(corpus, k)].user)//4:>8,}' for k in CONDITIONS]
    print(f'{corpus:<18} ' + ' '.join(row))

  [cached] prompt_payloads.json  (written 2026-08-08T15:18:02+00:00)
corpus                   A2       A3       A4       A5   (approx tokens)
------------------------------------------------------------
bank_marketing          351      808    1,838    3,286
diabetes_130us          566    2,724    4,371   10,049
online_retail_ii        323      557    1,653    2,704
nyc_tlc_yellow          402      982    2,640    5,087


## 9 · Verify the ladder is strictly nested

Each rung must contain everything the schema rung had, plus its own addition.
If A4 is not larger than A2, the census is not reaching the prompt.

In [ ]:
ok = True
for corpus in facts:
    u = {k: prompts[(corpus, k)].user for k in CONDITIONS}
    checks = {
        'A2 < A3': len(u['A2']) < len(u['A3']),
        'A2 < A4': len(u['A2']) < len(u['A4']),
        'A4 < A5': len(u['A4']) < len(u['A5']),
        'A4 has census': 'Profile computed on' in u['A4'],
        'A3 has no census': 'Profile computed on' not in u['A3'],
        'A5 has census': 'Profile computed on' in u['A5'],
        'all have schema': all('Columns:' in v for v in u.values()),
    }
    bad = [k for k, v in checks.items() if not v]
    ok &= not bad
    print(f'{"OK " if not bad else "!! "}{corpus:<18}' + (f'  failed: {bad}' if bad else ''))

# every prompt hash must be distinct
hashes = {(c, k): prompts[(c, k)].sha256 for c in facts for k in CONDITIONS}
dupes = len(hashes) - len(set(hashes.values()))
print(f'\n{len(hashes)} prompts, {len(set(hashes.values()))} distinct hashes, {dupes} collisions')
assert dupes == 0, 'prompt hash collision — two conditions produced identical text'
print('All ladders nested.' if ok else '!! LADDER PROBLEM — do not proceed to N2.')

OK bank_marketing    
OK diabetes_130us    
OK online_retail_ii  
OK nyc_tlc_yellow    

16 prompts, 16 distinct hashes, 0 collisions
All ladders nested.


## 10 · Read an actual prompt

This is the text a model will receive. Read it properly — this is your last
checkpoint before spending tokens, and a problem spotted here costs minutes
rather than a whole sweep.

In [ ]:
SHOW_CORPUS = 'bank_marketing'
SHOW_COND   = 'A4'

b = prompts[(SHOW_CORPUS, SHOW_COND)]
print('SYSTEM:'); print(b.system)
print('\n' + '=' * 72); print(f'USER ({SHOW_COND}, {SHOW_CORPUS}):'); print('=' * 72)
print(b.user)

SYSTEM:
You are a data quality engineer. Given a description of a table, author validation rules that identify genuinely defective records. Emit only JSON conforming to the provided schema, with no prose outside the JSON. Do not author rules for columns not listed. Prefer rules that would reject records a domain expert would consider erroneous, and avoid rules that would reject records that are merely unusual or that use a documented encoding convention.

USER (A4, bank_marketing):
Table: bank_marketing

Columns:
  age: int
  job: string
  marital: string
  education: string
  default: string
  balance: int
  housing: string
  loan: string
  contact: string
  day: int
  month: string
  duration: int
  campaign: int
  pdays: int
  previous: int
  poutcome: string
  y: string

Profile computed on an authoring sample of 9,042 rows.
All figures below are observed counts and frequencies.

  age (int): null 0.00%; 70 distinct; range 18 to 95; quantiles p01=23, p25=33, p50=39, p75=49, p99=71


## 11 · Compare A2 against A4 for one column

The clearest way to see what the census actually adds.

In [ ]:
COL = 'pdays'
for cond in CONDITIONS:
    print(f'--- {cond} ---')
    text = prompts[('bank_marketing', cond)].user
    lines = text.split('\n')
    for i, line in enumerate(lines):
        if COL in line:
            print(line)
            for extra in lines[i+1:i+7]:
                if extra.strip().startswith(('most frequent', "'", '-', '0','1','2','3','4','5','6','7','8','9')):
                    print(extra)
                else:
                    break
    print()

--- A2 ---
  pdays: int

--- A3 ---
  pdays: int
  pdays: Number of days since the client was last contacted in a previous campaign (numeric). A value of -1 means the client was not previously contacted.

--- A4 ---
  pdays: int
  pdays (int): null 0.00%; 376 distinct; range -1 to 828; quantiles p01=-1, p25=-1, p50=-1, p75=-1, p99=370
      most frequent values:
        -1: 7,447 (82.36%)
        182: 29 (0.32%)
        92: 27 (0.30%)
        181: 24 (0.27%)
        91: 23 (0.25%)

--- A5 ---
  pdays: int
  pdays (int): null 0.00%; 376 distinct; range -1 to 828; quantiles p01=-1, p25=-1, p50=-1, p75=-1, p99=370
      most frequent values:
        -1: 7,447 (82.36%)
        182: 29 (0.32%)
        92: 27 (0.30%)
        181: 24 (0.27%)
        91: 23 (0.25%)
  {"age": 46, "job": "services", "marital": "married", "education": "primary", "default": "no", "balance": 179, "housing": "yes", "loan": "no", "contact": "unknown", "day": 5, "month": "may", "duration": 1778, "campaign": 1, "pdays"

## 12 · Provenance and status

In [ ]:
def build_provenance():
    return {
        'notebook': 'P2_N1_census_and_payloads',
        'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'census_version': C.CENSUS_VERSION,
        'prompt_version': P.PROMPT_VERSION,
        'docs_version': D.DOCS_VERSION,
        'sample_seed': SAMPLE_SEED,
        'conditions': list(CONDITIONS),
        'doc_coverage': cov,
        'prompt_hashes': {f'{c}|{k}': prompts[(c, k)].sha256
                          for c in facts for k in CONDITIONS},
        'prompt_sizes_chars': {f'{c}|{k}': len(prompts[(c, k)].user)
                               for c in facts for k in CONDITIONS},
    }

prov, _ = ckpt.step('n1_provenance', 'json', build_provenance,
                    code_version=P.PROMPT_VERSION)
(ARTIFACTS / 'N1_provenance.json').write_text(json.dumps(prov, indent=2, default=str))
print('wrote', ARTIFACTS / 'N1_provenance.json')
print()
ckpt.status()
print()
print('N1 complete. Next: N2 — the generation sweep.')

  [cached] n1_provenance.json  (written 2026-08-08T15:18:02+00:00)
wrote /content/drive/MyDrive/Paper2_RuleAuthorship/artifacts/N1_provenance.json

Checkpoints in /content/drive/MyDrive/Paper2_RuleAuthorship/checkpoints
  OK  a3_doc_coverage.json                     708 B
  OK  census_bank_marketing.json               15,728 B
  OK  census_diabetes_130us.json               42,999 B
  OK  census_nyc_tlc_yellow.json               21,889 B
  OK  census_online_retail_ii.json             12,491 B
  OK  n0_provenance.json                       4,816 B
  OK  n1_provenance.json                       3,126 B
  OK  n3_rule_corpus.json                      4,172,519 B
  OK  n3_scored_rulesets.json                  164,524 B
  OK  n4_baselines.json                        3,785 B
  OK  n4_downstream_results.json               296,476 B
  OK  prompt_payloads.json                     176,012 B
  OK  source_file_hashes.json                  704 B
  OK  split_bank_marketing.json                486,750 